In [1]:
import re
import manifpy
import dash
from dash import html, dcc, Input, Output, State, no_update
import dash_cytoscape as cyto
import numpy as np
from scipy.linalg import block_diag
from collections import defaultdict

# ==== GBP import ====
from gbp.gbp import *

app = dash.Dash(__name__)
app.title = "Factor Graph SVD Abs&Recovery"

# -----------------------
# SLAM-like base graph
# -----------------------
def make_slam_like_graph(
    N=100, step_size=25, loop_prob=0.05, loop_radius=50, prior_prop=0.0, seed=None
):
    if seed is None:
        rng = np.random.default_rng()
    else:
        rng = np.random.default_rng(seed)
    # --- helpers ---
    def wrap_angle(a):
        # (-pi, pi]
        return np.arctan2(np.sin(a), np.cos(a))

    def relpose_SE2(pose_i, pose_j):
        """ Return z_ij = [dx_local, dy_local, dtheta] where 'local' is frame of i """
        xi, yi, thi = pose_i
        xj, yj, thj = pose_j
        Ri = np.array([[np.cos(thi), -np.sin(thi)],
                       [np.sin(thi),  np.cos(thi)]])
        dp = np.array([xj - xi, yj - yi])
        trans_local = Ri.T @ dp
        dth = wrap_angle(thj - thi)
        return np.array([trans_local[0], trans_local[1], dth], dtype=float)

    # --- SE(2) trajectory (smooth heading) ---
    poses = []
    x, y, th = 0.0, 0.0, 0.0
    poses.append((x, y, th))

    TURN_STD = 1  # rad, per step (tune smaller/larger as needed)
    for _ in range(1, int(N)):
        dth = rng.normal(0.0, TURN_STD)
        th = wrap_angle(th + dth)
        x += float(step_size) * np.cos(th)
        y += float(step_size) * np.sin(th)
        poses.append((x, y, th))

    # --- nodes (dim:3); visualization uses x,y ---
    nodes = []
    for i, (px, py, pth) in enumerate(poses):
        nodes.append({
            "data": {"id": f"{i}", "layer": 0, "dim": 3, "theta": float(pth), "num_base": 1},
            "position": {"x": float(px), "y": float(py)}  # for plotting only
        })

    # --- sequential odometry edges; attach measurement z_ij (local frame) ---
    edges = []
    for i in range(int(N) - 1):
        z_ij = relpose_SE2(poses[i], poses[i+1])
        edges.append({
            "data": {"source": f"{i}", "target": f"{i+1}", "kind": "odom", "z": z_ij.tolist()}
        })

    # --- loop-closure edges (proximity-triggered); also attach SE(2) measurements ---
    for i in range(int(N)):
        xi, yi, _ = poses[i]
        for j in range(i + 5, int(N)):  # consider loop closures only when gap >= 5 steps
            if rng.random() < float(loop_prob):
                xj, yj, _ = poses[j]
                if np.hypot(xi - xj, yi - yj) < float(loop_radius):
                    z_ij = relpose_SE2(poses[i], poses[j])
                    edges.append({
                        "data": {"source": f"{i}", "target": f"{j}", "kind": "loop", "z": z_ij.tolist()}
                    })

    # --- strong priors (anchors, etc.); still connect to the virtual "prior" ---
    if prior_prop <= 0.0:
        strong_ids = {0}
    elif prior_prop >= 1.0:
        strong_ids = set(range(N))
    else:
        k = max(1, int(np.floor(prior_prop * N)))
        strong_ids = set(rng.choice(N, size=k, replace=False).tolist())

    for i in strong_ids:
        edges.append({"data": {"source": f"{i}", "target": "prior", "kind": "prior"}})

    return nodes, edges



# -----------------------
# Initialization & Boundary
# -----------------------
def init_layers(N=100, step_size=25, loop_prob=0.05, loop_radius=50, prior_prop=0.0, seed=None):
    base_nodes, base_edges = make_slam_like_graph(N, step_size, loop_prob, loop_radius, prior_prop, seed)
    return [{"name": "base", "nodes": base_nodes, "edges": base_edges}]

VIEW_W, VIEW_H = 960, 600
ASPECT = VIEW_W / VIEW_H
AXIS_PAD=20.0

def adjust_bounds_to_aspect(xmin, xmax, ymin, ymax, aspect):
    cx=(xmin+xmax)/2; cy=(ymin+ymax)/2
    dx=xmax-xmin; dy=ymax-ymin
    if dx<=0: dx=1
    if dy<=0: dy=1
    if dx/dy > aspect:
        dy_new=dx/aspect
        return xmin,xmax,cy-dy_new/2,cy+dy_new/2
    else:
        dx_new=dy*aspect
        return cx-dx_new/2,cx+dx_new/2,ymin,ymax

def reset_global_bounds(base_nodes):
    global GLOBAL_XMIN, GLOBAL_XMAX, GLOBAL_YMIN, GLOBAL_YMAX
    global GLOBAL_XMIN_ADJ, GLOBAL_XMAX_ADJ, GLOBAL_YMIN_ADJ, GLOBAL_YMAX_ADJ
    xs=[n["position"]["x"] for n in base_nodes] or [0.0]
    ys=[n["position"]["y"] for n in base_nodes] or [0.0]
    GLOBAL_XMIN,GLOBAL_XMAX=min(xs),max(xs)
    GLOBAL_YMIN,GLOBAL_YMAX=min(ys),max(ys)
    GLOBAL_XMIN_ADJ,GLOBAL_XMAX_ADJ,GLOBAL_YMIN_ADJ,GLOBAL_YMAX_ADJ=adjust_bounds_to_aspect(
        GLOBAL_XMIN,GLOBAL_XMAX,GLOBAL_YMIN,GLOBAL_YMAX,ASPECT)

# ==== Blobal Status ====
layers = init_layers()
pair_idx = 0
reset_global_bounds(layers[0]["nodes"])
gbp_graph = None

# -----------------------
# GBP Graph Construction
# -----------------------
def build_noisy_pose_graph(
    nodes,
    edges,
    prior_sigma: float | tuple | np.ndarray = 1.0,   # A scalar expands to [s, s, s*THETA_RATIO]
    odom_sigma:  float | tuple | np.ndarray = 1.0,   # Same as above
    loop_sigma:  float | tuple | np.ndarray = 1.0,   # Same as above
    tiny_prior: float = 1e-10,
    seed=None,
):
    """
    Build an SE(2) 2D pose graph (x, y, theta). Binary edges use the SE(2) between nonlinear measurement model.
    Initialization policy: first propagate mu sequentially (reset when encountering a prior), then linearize all factors.
    """

    if seed is None:
        rng = np.random.default_rng()
    else:
        rng = np.random.default_rng(seed)

    THETA_RATIO = 0.01  # A scalar expands to [s, s, s*0.01]

    def _sigma_vec(s):
        v = np.array(s, dtype=float).ravel()
        if v.size == 1:
            s = float(v.item())
            return np.array([s, s, s * THETA_RATIO], dtype=float)
        assert v.size == 3, "If sigma is not scalar, it must be length-3 (x,y,theta)."
        return v

    def wrap_angle(a):
        return np.arctan2(np.sin(a), np.cos(a))

    def relpose_SE2(pose_i, pose_j):
        """z_ij = [dx_local, dy_local, dtheta], defined in i's coordinate frame."""
        xi, yi, thi = pose_i
        xj, yj, thj = pose_j
        c, s = np.cos(thi), np.sin(thi)
        RT = np.array([[ c, s],
                       [-s, c]])    # R(thi)^T
        dp = np.array([xj - xi, yj - yi])
        trans_local = RT @ dp
        dth = wrap_angle(thj - thi)
        return np.array([trans_local[0], trans_local[1], dth], dtype=float)

    def compose_SE2(pose_i, z_ij):
        """Given local measurement z_ij, propagate from pose_i to pose_j."""
        xi, yi, thi = pose_i
        dx, dy, dth = z_ij
        c, s = np.cos(thi), np.sin(thi)
        t_global = np.array([c*dx - s*dy, s*dx + c*dy])
        xj = xi + t_global[0]
        yj = yi + t_global[1]
        thj = wrap_angle(thi + dth)
        return np.array([xj, yj, thj], dtype=float)

    # ---------- Graph & variables ----------
    fg = FactorGraph(nonlinear_factors=False, eta_damping=0)

    var_nodes = []
    for i, n in enumerate(nodes):
        v = VariableNode(i, dofs=3)
        th = float(n["data"].get("theta", 0.0))  # Written by make_slam_like_graph (radians)
        v.GT = np.array([n["position"]["x"], n["position"]["y"], th], dtype=float)
        # Initialize mu to zero; will be set by sequential measurements later
        v.mu = np.zeros(3, dtype=float)
        var_nodes.append(v)

    fg.var_nodes = var_nodes
    fg.n_var_nodes = len(var_nodes)

    # ---------- Measurement model (SE(2) between) ----------
    def meas_fn_se2_between(xij, *args):
        # xij = [xi, yi, thi, xj, yj, thj]
        xi, yi, thi, xj, yj, thj = xij
        c, s = np.cos(thi), np.sin(thi)
        RT = np.array([[ c, s],
                       [-s, c]])
        dp = np.array([xj - xi, yj - yi])
        r  = RT @ dp
        dth = wrap_angle(thj - thi)
        return np.array([r[0], r[1], dth], dtype=float)
    
    def jac_fn_se2_between(xij, *args):
        # J: 3x6 w.r.t [x1,y1,th1, x2,y2,th2]
        x1, y1, th1, x2, y2, th2 = xij

        c1, s1 = np.cos(th1), np.sin(th1)
        c2, s2 = np.cos(th2), np.sin(th2)

        R1 = np.array([[c1, -s1],
                    [s1,  c1]], dtype=float)
        R2 = np.array([[c2, -s2],
                    [s2,  c2]], dtype=float)

        # p = R2^T (t1 - t2)
        t1 = np.array([x1, y1], dtype=float)
        t2 = np.array([x2, y2], dtype=float)
        p  = R2.T @ (t1 - t2)                 # shape (2,)

        # Rot(-pi/2) p = [ p_y, -p_x ]
        p_perp = np.array([p[1], -p[0]], dtype=float)

        H1 = np.zeros((3, 3), dtype=float)
        H1[0:2, 0:2] = R2.T @ R1
        H1[0:2, 2]   = p_perp
        H1[2, 2]     = 1.0
        H1 = -H1

        H2 = np.eye(3, dtype=float)

        J = np.zeros((3, 6), dtype=float)
        J[:, 0:3] = H1
        J[:, 3:6] = H2
        return J

    def meas_fn_unary(x, *args):
        return x

    def jac_fn_unary(x, *args):
        return np.eye(3, dtype=float)

    # ---------- Information matrices ----------
    prior_sigma_vec = _sigma_vec(prior_sigma)
    odom_sigma_vec  = _sigma_vec(odom_sigma)
    loop_sigma_vec  = _sigma_vec(loop_sigma)

    Lambda_prior = np.diag(1.0 / (prior_sigma_vec**2))
    Lambda_odom  = np.diag(1.0 / (odom_sigma_vec**2))
    Lambda_loop  = np.diag(1.0 / (loop_sigma_vec**2))
    
    # ---------- Factors; first create noisy measurements (no linearization yet) ----------
    odom_meas = {}   # (i,j) -> z_noisy
    prior_meas = {}  # i -> z_noisy (unary)
    factors = []
    fid = 0

    # Strong anchor: fix global reference (optionally anchor only x,y by relaxing Lambda_anchor[2,2])
    v0 = var_nodes[0]
    z_anchor = v0.GT.copy()
    Lambda_anchor = np.diag(1.0 / (np.array([1e-3, 1e-3, 1e-5])**2))
    f0 = Factor(fid, [v0], z_anchor, Lambda_anchor, meas_fn_unary, jac_fn_unary)
    f0.type = "prior"
    f0.compute_factor(linpoint=z_anchor, update_self=True)
    factors.append(f0)
    v0.adj_factors.append(f0)
    fid += 1

    for e in edges:
        src = e["data"]["source"]
        dst = e["data"]["target"]

        if dst != "prior":
            i, j = int(src), int(dst)

            # Ground-truth relative pose
            if "z" in e["data"]:
                z = np.array(e["data"]["z"], dtype=float).ravel()
            else:
                z = relpose_SE2(var_nodes[i].GT, var_nodes[j].GT)

            kind = e["data"].get("kind", "between")

            # >>> CHANGED: choose noise model per kind
            if kind == "loop":
                noise_vec = rng.normal(0.0, loop_sigma_vec, size=3)
                this_Lambda = Lambda_loop
            else:
                # default treat as odom
                noise_vec = rng.normal(0.0, odom_sigma_vec, size=3)
                this_Lambda = Lambda_odom

            z_noisy = z.copy()
            z_noisy[:2] += noise_vec[:2]
            z_noisy[2]  = wrap_angle(z_noisy[2] + noise_vec[2])

            # only store sequential odom for init mu forward-prop
            if kind == "odom" and (j == i + 1):
                odom_meas[(i, j)] = z_noisy

            vi, vj = var_nodes[i], var_nodes[j]
            f = Factor(fid, [vi, vj], z_noisy, this_Lambda, meas_fn_se2_between, jac_fn_se2_between)
            f.type = kind
            factors.append(f)
            vi.adj_factors.append(f)
            vj.adj_factors.append(f)
            fid += 1

        else:
            # Prior edge: create a noisy measurement for this variable
            i = int(src)
            z = var_nodes[i].GT.copy()
            noise = rng.normal(0.0, prior_sigma_vec, size=3)
            z[:2] += noise[:2]
            z[2]   = wrap_angle(z[2] + noise[2])
            prior_meas[i] = z

            vi = var_nodes[i]
            f = Factor(fid, [vi], z, Lambda_prior, meas_fn_unary, jac_fn_unary)
            f.type = "prior"
            factors.append(f)
            vi.adj_factors.append(f)
            fid += 1

    # ---------- Sequentially initialize mu: forward propagation from node 0; reset when hitting a prior ----------
    N = len(var_nodes)
    # Start: use GT


    var_nodes[0].linpoint = var_nodes[0].GT

    for i in range(N - 1):
        # First propagate via odometry
        if (i, i+1) in odom_meas:
            var_nodes[i+1].linpoint = compose_SE2(var_nodes[i].linpoint, odom_meas[(i, i+1)])
        else:
            # If the edge is missing, fall back to GT
            var_nodes[i+1].linpoint = var_nodes[i+1].GT.copy()

        # If i+1 has a prior, override with the prior (replace the propagated value)
        if (i + 1) in prior_meas:
            var_nodes[i+1].linpoint = prior_meas[i + 1].copy()

    # ---------- Linearize all factors (mu is now in place) ----------
    for f in factors:
        lin = np.concatenate([vn.linpoint for vn in f.adj_var_nodes]) if f.adj_var_nodes else np.array([])
        f.compute_factor(linpoint=lin, update_self=True)

    fg.factors = factors
    fg.n_factor_nodes = len(factors)
    return fg


def wrap_angle(a: float) -> float:
    return float(np.arctan2(np.sin(a), np.cos(a)))

def relpose_SE2(pose_i, pose_j):
    """z_ij = [dx_local, dy_local, dtheta], defined in i's coordinate frame."""
    xi, yi, thi = pose_i
    xj, yj, thj = pose_j
    c, s = np.cos(thi), np.sin(thi)
    RT = np.array([[ c, s],
                    [-s, c]])    # R(thi)^T
    dp = np.array([xj - xi, yj - yi])
    trans_local = RT @ dp
    dth = wrap_angle(thj - thi)
    return np.array([trans_local[0], trans_local[1], dth], dtype=float)

def compose_SE2(pose_i, z_ij):
    """Given local measurement z_ij, propagate from pose_i to pose_j."""
    xi, yi, thi = pose_i
    dx, dy, dth = z_ij
    c, s = np.cos(thi), np.sin(thi)
    t_global = np.array([c*dx - s*dy, s*dx + c*dy])
    xj = xi + t_global[0]
    yj = yi + t_global[1]
    thj = wrap_angle(thi + dth)
    return np.array([xj, yj, thj], dtype=float)


def energy_loss(graph, include_priors: bool = True, include_factors: bool = True) -> float:
    """
    It is actually the sum of squares of distances.
    """
    total = 0.0

    for f in graph.factors[:graph.n_factor_nodes]:
        J = f.jac_fn(f.linpoint, *f.args)
        mu_est = f.linpoint
        for i,v in enumerate(f.adj_var_nodes):
            mu_est[i*3:i*3+3] = compose_SE2(mu_est[i*3:i*3+3], v.mu)
        pred_measurement = f.meas_fn(mu_est, *f.args)
        r = relpose_SE2(f.measurement, pred_measurement)

        total += 0.5 * float(r.T @ f.measurement_lambda @ r)
    return total

In [2]:
def max_lam_change(gbp_graph, prev_lams):
    """
    Compute max change of belief.lam over all variables.

    Parameters
    ----------
    gbp_graph : FactorGraph
        Graph with var_nodes[i].belief.lam (numpy array)
    prev_lams : list[np.ndarray]
        Previous iteration lams, same order as var_nodes

    Returns
    -------
    float
        Maximum absolute change over all variables
    """
    max_change = 0.0

    if len(prev_lams) != len(gbp_graph.var_nodes):
        raise ValueError("prev_lams size mismatch")

    for i, v in enumerate(gbp_graph.var_nodes):
        if v is None:
            continue

        lam_now = v.belief.lam
        lam_prev = prev_lams[i]

        diff = np.max(np.abs(lam_now - lam_prev))
        max_change = max(max_change, diff)

    return max_change

def max_eta_change(gbp_graph, prev_etas):
    """
    Compute max change of belief.eta over all variables.

    Parameters
    ----------
    gbp_graph : FactorGraph
        Graph with var_nodes[i].belief.eta (numpy array)
    prev_etas : list[np.ndarray]
        Previous iteration etas, same order as var_nodes

    Returns
    -------
    float
        Maximum absolute change over all variables
    """
    max_change = 0.0

    if len(prev_etas) != len(gbp_graph.var_nodes):
        raise ValueError("prev_etas size mismatch")

    for i, v in enumerate(gbp_graph.var_nodes):
        if v is None:
            continue

        eta_now = v.belief.eta
        eta_prev = prev_etas[i]

        diff = np.max(np.abs(eta_now - eta_prev))
        max_change = max(max_change, diff)

    return max_change


def energy_map(graph, include_priors: bool = True, include_factors: bool = True) -> float:
    """
    It is actually the sum of squares of distances.
    """
    total = 0.0

    for v in graph.var_nodes[:graph.n_var_nodes]:
        gt = np.asarray(v.GT[0:2], dtype=float)
        r = np.asarray(v.mu[0:2], dtype=float) - gt
        total += 0.5 * float(r.T @ r)

    return total


N=512
step=25
prob=0.05
radius=50 
prior_prop=0.02
prior_sigma=1
odom_sigma=1
loop_sigma=10
layers = []



layers = init_layers(N=N, step_size=step, loop_prob=prob, loop_radius=radius, prior_prop=prior_prop, seed=2001)
pair_idx = 0


# 构建 GBP 图
gbp_graph = build_noisy_pose_graph(layers[0]["nodes"], layers[0]["edges"],
                                    prior_sigma=prior_sigma,
                                    odom_sigma=odom_sigma,
                                    loop_sigma=loop_sigma,
                                    seed=2001)
gbp_graph.num_undamped_iters = 0
gbp_graph.min_linear_iters = 20000000
opts=[{"label":"base","value":"base"}]
print(energy_loss(gbp_graph, include_priors=True, include_factors=True))

for i in range(1):
    delta, Sigma = gbp_graph.joint_distribution_cov()
    for i, v in enumerate(gbp_graph.var_nodes):
        gbp_graph.var_nodes[i].mu = delta[i*3:(i+1)*3]
    print(energy_loss(gbp_graph, include_priors=True, include_factors=True))

    """
    for i, v in enumerate(gbp_graph.var_nodes):
        gbp_graph.var_nodes[i].linpoint = compose_SE2(gbp_graph.var_nodes[i].linpoint, 0.1*delta[i*3:(i+1)*3])

    for f in gbp_graph.factors:
        lin = np.concatenate([vn.linpoint for vn in f.adj_var_nodes]) if f.adj_var_nodes else np.array([])
        f.compute_factor(linpoint=lin, update_self=True)
    """




1846.74404520422
112.07555155132799


In [6]:
N=512
step=25
prob=0.05
radius=50 
prior_prop=0.02
prior_sigma=1
odom_sigma=1
loop_sigma=10
layers = []



layers = init_layers(N=N, step_size=step, loop_prob=prob, loop_radius=radius, prior_prop=prior_prop, seed=2001)
pair_idx = 0


# 构建 GBP 图
gbp_graph = build_noisy_pose_graph(layers[0]["nodes"], layers[0]["edges"],
                                    prior_sigma=prior_sigma,
                                    odom_sigma=odom_sigma,
                                    loop_sigma=loop_sigma,
                                    seed=2001)
layers[0]["graph"] = gbp_graph
gbp_graph.num_undamped_iters = 0
gbp_graph.min_linear_iters = 2000000
opts=[{"label":"base","value":"base"}]

basegraph = layers[0]["graph"]
print(energy_loss(basegraph, include_priors=True, include_factors=True))


for k in range(1):
    prev_lams = [v.belief.lam.copy() for v in basegraph.var_nodes]
    prev_etas = [v.belief.eta.copy() for v in basegraph.var_nodes]
    max_lam_change_values = []
    max_eta_change_values = []
    for it in range(100):
        basegraph.synchronous_iteration()

        max_lam_change_value = max_lam_change(basegraph, prev_lams)
        max_eta_change_value = max_eta_change(basegraph, prev_etas)
        max_lam_change_values.append(max_lam_change_value)
        max_eta_change_values.append(max_eta_change_value)
        
        energy = energy_loss(basegraph, include_priors=True, include_factors=True)
        #print(f"OuterIter {k}, InnerIter {it:03d}, MaxLamChange {max_lam_change_value:.4e}, MaxEtaChange {max_eta_change_value:.4e} | Energy = {energy:.6f}")
        print(basegraph.var_nodes[500].mu)
        prev_lams = [v.belief.lam.copy() for v in basegraph.var_nodes]
        prev_etas = [v.belief.eta.copy() for v in basegraph.var_nodes]

    



1846.74404520422
[5.96737052 7.30450694 0.06871848]
[-2.03076658 -1.93649629 -0.34818435]
[-7.37080037  4.94748007 -0.08055203]
[-7.13949626  3.70366841 -0.07493535]
[-7.19698702  4.54625311 -0.07907771]
[-6.96159719  3.69333957 -0.07588072]
[-7.37769936  3.98028762 -0.07402779]
[-21.29957012   7.1241393   -0.05743041]
[-6.54486752  3.72810341 -0.07122386]
[45.4744247  -8.13695596 -0.16819245]
[-6.26406643  3.40357525 -0.06802538]
[-1.66967741  2.60347714 -0.09845052]
[-7.04237752 -0.75959737 -0.1032066 ]
[-8.99444711  9.62496697 -0.10703448]
[-6.42272213  4.04644874 -0.16466086]
[-18.36515269  -5.04534023  -0.03764213]
[-12.11413426  -8.20278653   0.02314351]
[-5.23301791 -5.64716192 -0.19539437]
[-62.13951919 -87.18512993   1.39893268]
[116.07417562 -16.66277054  -2.32914248]
[-56.67630619 -65.21422202   1.21806299]
[153.77415874 -40.26116313  -3.02236414]
[-65.88670094 -47.58983612   1.25817239]
[229.04074699 -89.89674507  -4.16966817]
[-57.4758512  -42.42336932   1.15109277]
[185.9

In [ ]:
def jac_fn_se2_between(xij, *args):
    # J: 3x6 w.r.t [x1,y1,th1, x2,y2,th2]
    x1, y1, th1, x2, y2, th2 = xij

    c1, s1 = np.cos(th1), np.sin(th1)
    c2, s2 = np.cos(th2), np.sin(th2)

    R1 = np.array([[c1, -s1],
                [s1,  c1]], dtype=float)
    R2 = np.array([[c2, -s2],
                [s2,  c2]], dtype=float)

    # p = R2^T (t1 - t2)
    t1 = np.array([x1, y1], dtype=float)
    t2 = np.array([x2, y2], dtype=float)
    p  = R2.T @ (t1 - t2)                 # shape (2,)

    # Rot(-pi/2) p = [ p_y, -p_x ]
    p_perp = np.array([p[1], -p[0]], dtype=float)

    H1 = np.zeros((3, 3), dtype=float)
    H1[0:2, 0:2] = R2.T @ R1
    H1[0:2, 2]   = p_perp
    H1[2, 2]     = 1.0
    H1 = -H1

    H2 = np.eye(3, dtype=float)

    J = np.zeros((3, 6), dtype=float)
    J[:, 0:3] = H1
    J[:, 3:6] = H2
    return J




gbp_graph = build_noisy_pose_graph(layers[0]["nodes"], layers[0]["edges"],
                                    prior_sigma=prior_sigma,
                                    odom_sigma=odom_sigma,
                                    loop_sigma=loop_sigma,
                                    seed=2001)
layers[0]["graph"] = gbp_graph
gbp_graph.num_undamped_iters = 0
gbp_graph.min_linear_iters = 20000
opts=[{"label":"base","value":"base"}]

basegraph = layers[0]["graph"]
basegraph.relinearise_factors()
a = np.diag([1.0, 1.0, 100])
print(a)
A = a@jac_fn_se2_between(np.concatenate([basegraph.factors[100].adj_var_nodes[0].linpoint, basegraph.factors[100].adj_var_nodes[1].linpoint]))
J = A.T@A
print(np.array2string(J, suppress_small=True, precision=6))






[[  1.   0.   0.]
 [  0.   1.   0.]
 [  0.   0. 100.]]
[[     1.           -0.           -5.660181     -0.981757      0.190139
       0.      ]
 [    -0.            1.           24.514491     -0.190139     -0.981757
       0.      ]
 [    -5.660181     24.514491  10632.997924      0.895774    -25.143498
  -10000.      ]
 [    -0.981757     -0.190139      0.895774      1.            0.
       0.      ]
 [     0.190139     -0.981757    -25.143498      0.            1.
       0.      ]
 [     0.            0.       -10000.            0.            0.
   10000.      ]]


In [ ]:
basegraph.factors[100].adj_var_nodes[0].variableID

99

In [ ]:
"""
gtsam_se2_benchmark_lm.py

Purpose
-------
Pure GTSAM benchmark for nonlinear SE(2) pose-graph:
- Use GTSAM's Pose2 / BetweenFactorPose2 / PriorFactorPose2
- Build one full graph (prior + odom + loop closures)
- Solve once with Levenberg-Marquardt (batch, one-shot)
- Report build time + solve time + basic diagnostics

Notes
-----
1) This is NOT using your GBP classes at all.
2) Measurements are SE(2) relative motions z_ij = (dx_local, dy_local, dtheta)
   which matches GTSAM Pose2 "between" convention.
3) Noise injection is done once, deterministically by seed, at graph construction.
4) Anchor: strong prior at node 0, plus optionally additional priors controlled by prior_prop.
   (Matches your generator's "prior edges to virtual prior".)

Run
---
python gtsam_se2_benchmark_lm.py
"""

import time
import numpy as np
import gtsam
from gtsam import symbol


# -----------------------
# Utilities
# -----------------------
def wrap_angle(a: float) -> float:
    return float(np.arctan2(np.sin(a), np.cos(a)))


def sigma_vec(s, theta_ratio=0.01) -> np.ndarray:
    v = np.array(s, dtype=float).ravel()
    if v.size == 1:
        s0 = float(v.item())
        return np.array([s0, s0, s0 * theta_ratio], dtype=float)
    if v.size != 3:
        raise ValueError("sigma must be scalar or length-3 [sx, sy, sth]")
    return v.astype(float)


def relpose_SE2(pose_i, pose_j):
    """z_ij = [dx_local, dy_local, dtheta], defined in i's coordinate frame."""
    xi, yi, thi = pose_i
    xj, yj, thj = pose_j
    c, s = np.cos(thi), np.sin(thi)
    RT = np.array([[ c, s],
                    [-s, c]])    # R(thi)^T
    dp = np.array([xj - xi, yj - yi])
    trans_local = RT @ dp
    dth = wrap_angle(thj - thi)
    return np.array([trans_local[0], trans_local[1], dth], dtype=float)

def compose_SE2(pose_i, z_ij):
    """Given local measurement z_ij, propagate from pose_i to pose_j."""
    xi, yi, thi = pose_i
    dx, dy, dth = z_ij
    c, s = np.cos(thi), np.sin(thi)
    t_global = np.array([c*dx - s*dy, s*dx + c*dy])
    xj = xi + t_global[0]
    yj = yi + t_global[1]
    thj = wrap_angle(thi + dth)
    return np.array([xj, yj, thj], dtype=float)


# -----------------------
# SLAM-like generator (clean GT + clean relative z on edges)
# -----------------------

def make_slam_like_graph(
    N=100, 
    step_size=25, 
    loop_prob=0.05, 
    loop_radius=50, 
    prior_prop=0.0, 
    seed=None
):
    if seed is None:
        rng = np.random.default_rng()
    else:
        rng = np.random.default_rng(seed)
    # --- helpers ---
    def wrap_angle(a):
        # (-pi, pi]
        return np.arctan2(np.sin(a), np.cos(a))

    def relpose_SE2(pose_i, pose_j):
        """ Return z_ij = [dx_local, dy_local, dtheta] where 'local' is frame of i """
        xi, yi, thi = pose_i
        xj, yj, thj = pose_j
        Ri = np.array([[np.cos(thi), -np.sin(thi)],
                       [np.sin(thi),  np.cos(thi)]])
        dp = np.array([xj - xi, yj - yi])
        trans_local = Ri.T @ dp
        dth = wrap_angle(thj - thi)
        return np.array([trans_local[0], trans_local[1], dth], dtype=float)

    # --- SE(2) trajectory (smooth heading) ---
    poses = []
    x, y, th = 0.0, 0.0, 0.0
    poses.append((x, y, th))

    TURN_STD = 1  # rad, per step (tune smaller/larger as needed)
    for _ in range(1, int(N)):
        dth = rng.normal(0.0, TURN_STD)
        th = wrap_angle(th + dth)
        x += float(step_size) * np.cos(th)
        y += float(step_size) * np.sin(th)
        poses.append((x, y, th))

    # --- nodes (dim:3); visualization uses x,y ---
    nodes = []
    for i, (px, py, pth) in enumerate(poses):
        nodes.append([px, py, pth])

    # --- sequential odometry edges; attach measurement z_ij (local frame) ---
    edges = []
    for i in range(int(N) - 1):
        z_ij = relpose_SE2(poses[i], poses[i+1])
        edges.append({
            "data": {"source": f"{i}", "target": f"{i+1}", "kind": "odom", "z": z_ij.tolist()}
        })

    # --- loop-closure edges (proximity-triggered); also attach SE(2) measurements ---
    for i in range(int(N)):
        xi, yi, _ = poses[i]
        for j in range(i + 5, int(N)):  # consider loop closures only when gap >= 5 steps
            if rng.random() < float(loop_prob):
                xj, yj, _ = poses[j]
                if np.hypot(xi - xj, yi - yj) < float(loop_radius):
                    z_ij = relpose_SE2(poses[i], poses[j])
                    edges.append({
                        "data": {"source": f"{i}", "target": f"{j}", "kind": "loop", "z": z_ij.tolist()}
                    })

    # --- strong priors (anchors, etc.); still connect to the virtual "prior" ---
    if prior_prop <= 0.0:
        strong_ids = {0}
    elif prior_prop >= 1.0:
        strong_ids = set(range(N))
    else:
        k = max(1, int(np.floor(prior_prop * N)))
        strong_ids = set(rng.choice(N, size=k, replace=False).tolist())

    for i in strong_ids:
        edges.append({"data": {"source": f"{i}", "target": "prior", "kind": "prior"}})

    return np.array(nodes), edges


# -----------------------
# Build GTSAM Pose2 graph + initial
# -----------------------
import numpy as np
import gtsam
from gtsam import symbol

def build_gtsam_pose2_graph(
    GT_poses,  # (N,3) used ONLY to generate clean measurements / prior means
    edges,     # list of (i, j, kind, z_clean)  z_clean = [dx,dy,dth] in i-local
    prior_sigma=1.0,
    odom_sigma=1.0,
    loop_sigma=1.0,
    theta_ratio=0.01,
    seed=2001,
    add_measurement_noise=True,
    add_prior_noise=True,
    anchor_strong=True,
    anchor_sigmas=(1e-3, 1e-3, 1e-5),
):
    rng = np.random.default_rng(seed)
    N = int(GT_poses.shape[0])

    # noise models (sigmas)
    prior_sig = sigma_vec(prior_sigma, theta_ratio)  # (3,)
    odom_sig  = sigma_vec(odom_sigma, theta_ratio)   # (3,)
    loop_sig  = sigma_vec(loop_sigma, theta_ratio)   # (3,)

    prior_model = gtsam.noiseModel.Diagonal.Sigmas(gtsam.Point3(*prior_sig))
    odom_model  = gtsam.noiseModel.Diagonal.Sigmas(gtsam.Point3(*odom_sig))
    loop_model  = gtsam.noiseModel.Diagonal.Sigmas(gtsam.Point3(*loop_sig))

    graph = gtsam.NonlinearFactorGraph()
    initial = gtsam.Values()

    def key(i: int):
        return symbol("x", int(i))

    # ------------------------------------------------------------
    # Store the EXACT noisy measurements used by factors
    # so init can reuse them (no GT init, deterministic).
    # ------------------------------------------------------------
    strong_prior_meas = {}   # i -> noisy Pose (x,y,th) used in PriorFactorPose2
    odom_meas = {}           # (i,i+1) -> noisy z_ij used in BetweenFactorPose2 (odom edges only)


    a = np.array(anchor_sigmas, dtype=float).ravel()
    if a.size != 3:
        raise ValueError("anchor_sigmas must be length-3")
    anchor_model = gtsam.noiseModel.Diagonal.Sigmas(
        gtsam.Point3(float(a[0]), float(a[1]), float(a[2]))
    )

    z0 = GT_poses[0].copy()
    graph.add(gtsam.PriorFactorPose2(
        key(0),
        gtsam.Pose2(float(z0[0]), float(z0[1]), float(z0[2])),
        anchor_model
    ))


    for e in edges:
        src = e["data"]["source"]
        dst = e["data"]["target"]

        if dst != "prior":
            i, j = int(src), int(dst)

            # Ground-truth relative pose
            if "z" in e["data"]:
                z = np.array(e["data"]["z"], dtype=float).ravel()
            else:
                z = relpose_SE2(GT_poses[i], GT_poses[j])

            kind = e["data"].get("kind", "between")

            if add_measurement_noise:
                if kind == "loop":
                    n = rng.normal(0.0, loop_sig, size=3)
                    model = loop_model
                else:
                    n = rng.normal(0.0, odom_sig, size=3)
                    model = odom_model

                z_noisy = z.copy()
                z_noisy[:2] += n[:2]
                z_noisy[2] = wrap_angle(z_noisy[2] + n[2])
            else:
                model = loop_model if kind == "loop" else odom_model
                z_noisy = z.copy()

            # store odom for chaining ONLY if it's sequential odom edge (i,i+1)
            if kind == "odom" and (j == i + 1):
                odom_meas[(i, j)] = z_noisy.copy()

            graph.add(gtsam.BetweenFactorPose2(
                key(i), key(j),
                gtsam.Pose2(float(z_noisy[0]), float(z_noisy[1]), float(z_noisy[2])),
                model
            ))

        else:
            i = int(src)
            z = GT_poses[i].copy()
            if add_prior_noise:
                n = rng.normal(0.0, prior_sig, size=3)
                z[:2] += n[:2]
                z[2] = wrap_angle(z[2] + n[2])

            strong_prior_meas[i] = z.copy()

            graph.add(gtsam.PriorFactorPose2(
                key(i),
                gtsam.Pose2(float(z[0]), float(z[1]), float(z[2])),
                prior_model
            ))

    # ------------------------------------------------------------
    # (D) INITIALIZATION per your requirement:
    # start from (0,0,0), propagate with odom ⊕,
    # when hit a strong prior node, hard reset to that prior mean measurement.
    # ------------------------------------------------------------
    mu = np.zeros((N, 3), dtype=float)
    mu[0] = np.array([0.0, 0.0, 0.0], dtype=float)

    for i in range(N - 1):
        if (i, i + 1) in odom_meas:
            mu[i + 1] = compose_SE2(mu[i], odom_meas[(i, i + 1)])
        else:
            print(f"Missing odom between {i} and {i+1}")
            # missing odom: hold last pose (still not GT)
            mu[i + 1] = mu[i].copy()

        # reset at strong prior node
        if (i + 1) in strong_prior_meas:
            mu[i + 1] = strong_prior_meas[i + 1].copy()

    for i in range(N):
        initial.insert(key(i), gtsam.Pose2(float(mu[i, 0]), float(mu[i, 1]), float(mu[i, 2])))

    
    return graph, initial



# -----------------------
# Solve once with LM (batch)
# -----------------------
def solve_lm_once(graph, initial, max_iters=100, rel_error_tol=1e-9, abs_error_tol=1e-9, lambda_initial=1e-3):
    params = gtsam.LevenbergMarquardtParams()
    #params = gtsam.GaussNewtonParams()
    params.setVerbosityLM("SILENT")  # for timing
    #params.setVerbosityLM("SUMMARY")
    params.setMaxIterations(int(max_iters))
    params.setRelativeErrorTol(float(rel_error_tol))
    params.setAbsoluteErrorTol(float(abs_error_tol))
    params.setlambdaInitial(float(lambda_initial))

    optimizer = gtsam.LevenbergMarquardtOptimizer(graph, initial, params)
    #optimizer = gtsam.GaussNewtonOptimizer(graph, initial, params)

    t0 = time.perf_counter()
    result = optimizer.optimize()
    t1 = time.perf_counter()
    return result, (t1 - t0)


def batch_benchmark(
    N=5000,
    step_size=25,
    loop_prob=0.05,
    loop_radius=50,
    prior_prop=0.02,
    prior_sigma=1.0,
    odom_sigma=1.0,
    loop_sigma=1.0,
    theta_ratio=0.01,
    seed=2001,
    runs=5,
    warmup=1,
):
    # generate GT + clean graph topology
    GT, edges = make_slam_like_graph(
        N=N,
        step_size=step_size,
        loop_prob=loop_prob,
        loop_radius=loop_radius,
        prior_prop=prior_prop,
        seed=seed,
    )

    # Build once (build time matters too, so measure separately if you want)
    t_build0 = time.perf_counter()
    graph, initial = build_gtsam_pose2_graph(
        GT, edges,
        prior_sigma=prior_sigma,
        odom_sigma=odom_sigma,
        loop_sigma=loop_sigma,
        theta_ratio=theta_ratio,
        seed=seed,
        add_measurement_noise=True,
        add_prior_noise=True,
        anchor_strong=True,
    )
    t_build1 = time.perf_counter()

    # Warmup solve(s) to avoid first-call overhead in timing (allocations, etc.)
    for _ in range(max(0, int(warmup))):
        _ = solve_lm_once(graph, initial, max_iters=100)[0]

    # Timed runs
    times = []
    last_result = None
    for _ in range(int(runs)):
        last_result, dt = solve_lm_once(graph, initial, max_iters=200)
        times.append(dt)

    times = np.array(times, dtype=float)

    # Basic diagnostics: final error
    final_error = graph.error(last_result)

    print("=== GTSAM Pose2 Batch LM Benchmark ===")
    print(f"sigmas: prior={prior_sigma}, odom={odom_sigma}, loop={loop_sigma}, theta_ratio={theta_ratio}")
    print(f"build_time: {(t_build1 - t_build0)*1000:.3f} ms")
    print(f"solve_time over {runs} runs (after warmup={warmup}):")
    print(f"  mean = {times.mean()*1000:.3f} ms")
    print(f"  std  = {times.std()*1000:.3f} ms")
    print(f"  min  = {times.min()*1000:.3f} ms")
    print(f"  max  = {times.max()*1000:.3f} ms")
    print(f"final_graph_error = {final_error:.6g}")

    lm_iter_timing_print(graph, initial, max_iters=200)
    return {
        "build_s": float(t_build1 - t_build0),
        "solve_s_mean": float(times.mean()),
        "solve_s_std": float(times.std()),
        "solve_s_min": float(times.min()),
        "solve_s_max": float(times.max()),
        "final_error": float(final_error),
        "edges": int(len(edges)),
        "GT": GT,
        "result": last_result,
        "initial": initial,
        "graph": graph,
    }


import time
import numpy as np
import gtsam
from gtsam import symbol

def lm_iter_timing_print(
    graph,
    initial,
    max_iters=200,
    lambda_initial=1e-3,
    abs_tol=1e-9,          # absolute error improvement threshold
    rel_tol=1e-9,          # relative error improvement threshold
    stall_iters=3,         # require k consecutive "small improvements" before stopping
    min_iters=1,           # force at least this many iterations
    print_every=1,
):
    """
    Run LM manually with per-iteration timing and early stopping based on error improvement.

    Stopping rule (after min_iters):
      let e_prev, e_now
      improvement = e_prev - e_now

      stop if improvement <= abs_tol  OR  improvement/|e_prev| <= rel_tol
      for `stall_iters` consecutive iterations.

    Notes
    -----
    - This is the cleanest Python-side way to mimic LM termination when verbosity is not available.
    - If an iteration increases error (rejected step / damping behavior), improvement becomes negative;
      that counts as "stall" and accelerates termination unless you change the logic below.
    """

    params = gtsam.GaussNewtonParams()
    params.setVerbosity("SILENT")          # 可选：SILENT / ERROR / TERMINATION / etc.
    params.setMaxIterations(int(max_iters))

    # 如果你想尽量不让它内部 early stop，就把阈值设得极小
    params.setRelativeErrorTol(1e-20)
    params.setAbsoluteErrorTol(1e-20)

    opt = gtsam.GaussNewtonOptimizer(graph, initial, params)
    print("init error",opt.error())  # 初始化内部状态

    ts = []
    errors = []

    e_prev = float(opt.error())
    errors.append(e_prev)

    stall_count = 0

    for k in range(max_iters):
        t0 = time.perf_counter()
        opt.iterate()  # 1 iter = relinearize + linear solve + retract (+ lambda update)
        t1 = time.perf_counter()

        dt_ms = (t1 - t0) * 1000.0
        ts.append(dt_ms)

        e_now = float(opt.error())
        errors.append(e_now)

        improvement = e_prev - e_now
        rel_impr = improvement / (abs(e_prev) + 1e-30)

        if (k % print_every) == 0:
            print(
                f"[TIMING] iter {k:03d}: {dt_ms:8.3f} ms | "
                f"error={e_now:.6e} | dE={improvement:.5e} | rel_dE={rel_impr:.5e}"
            )

        # --- early stop logic ---
        if (k + 1) >= min_iters:
            # define "stalled" as very small improvement (or worse)
            stalled = (improvement <= abs_tol) or (rel_impr <= rel_tol)
            if stalled:
                stall_count += 1
            else:
                stall_count = 0

            if stall_count >= stall_iters:
                print(
                    f"[STOP] stalled for {stall_iters} iters "
                    f"(abs_tol={abs_tol:.1e}, rel_tol={rel_tol:.1e}) at iter {k:03d}."
                )
                break

        e_prev = e_now

    return opt.values(), np.array(ts, dtype=float), np.array(errors, dtype=float)


if __name__ == "__main__":
    # ---- your config (copied from your message) ----
    stats = batch_benchmark(
        N=512,
        step_size=25,
        loop_prob=0.05,
        loop_radius=50,
        prior_prop=0.02,
        prior_sigma=1,
        odom_sigma=1,
        loop_sigma=10,
        theta_ratio=0.01,   # same as your THETA_RATIO
        seed=2001,
        runs=10,
        warmup=1,
    )



=== GTSAM Pose2 Batch LM Benchmark ===
sigmas: prior=1, odom=1, loop=10, theta_ratio=0.01
build_time: 12.032 ms
solve_time over 10 runs (after warmup=1):
  mean = 28.933 ms
  std  = 1.255 ms
  min  = 27.304 ms
  max  = 30.880 ms
final_graph_error = 111.287
init error 1846.7440452042213
[TIMING] iter 000:    2.064 ms | error=1.120756e+02 | dE=1.73467e+03 | rel_dE=9.39312e-01
[TIMING] iter 001:    1.987 ms | error=1.112882e+02 | dE=7.87401e-01 | rel_dE=7.02562e-03
[TIMING] iter 002:    2.009 ms | error=1.112873e+02 | dE=8.08698e-04 | rel_dE=7.26671e-06
[TIMING] iter 003:    1.955 ms | error=1.112873e+02 | dE=3.92487e-06 | rel_dE=3.52679e-08
[TIMING] iter 004:    1.874 ms | error=1.112873e+02 | dE=-4.52198e-07 | rel_dE=-4.06334e-09
[TIMING] iter 005:    1.749 ms | error=1.112873e+02 | dE=-6.48672e-09 | rel_dE=-5.82880e-11
[TIMING] iter 006:    1.866 ms | error=1.112873e+02 | dE=2.90711e-10 | rel_dE=2.61226e-12
[STOP] stalled for 3 iters (abs_tol=1.0e-09, rel_tol=1.0e-09) at iter 006.


In [10]:
import numpy as np

def global_Lam_eta_via_hessian(nonlinear_graph, values, ordering=None):
    linear = nonlinear_graph.linearize(values)
    if ordering is None:
        Lam, eta = linear.hessian()
    else:
        Lam, eta = linear.hessian(ordering)

    Lam = np.array(Lam, dtype=float)
    eta = np.array(eta, dtype=float).reshape(-1)
    return Lam, eta

Lam, eta = global_Lam_eta_via_hessian(stats["graph"], stats["initial"])

gbp_graph = build_noisy_pose_graph(layers[0]["nodes"], layers[0]["edges"],
                                    prior_sigma=prior_sigma,
                                    odom_sigma=odom_sigma,
                                    loop_sigma=loop_sigma,
                                    seed=2001)



eta2, Lam2 = gbp_graph.joint_distribution_inf()


delta = np.linalg.solve(Lam2, eta2)

print("difference of lam and eta: ", np.linalg.norm(Lam - Lam2), np.linalg.norm(eta - eta2))


for i, v in enumerate(gbp_graph.var_nodes):
    gbp_graph.var_nodes[i].mu = delta[i*3:(i+1)*3]
print(energy_loss(gbp_graph, include_priors=True, include_factors=True))



difference of lam and eta:  1.907348634746258e-06 1.2997562497857417e-11
112.07555155132822


In [8]:
basegraph.var_nodes[117].adj_factors[2].factorID

585

In [29]:
i = 585
f = stats["graph"].at(i)
gf = f.linearize(stats["initial"])

J = np.array(gf.getA(), dtype=float)
A = J.T@J
b= np.array(gf.getb(), dtype=float).reshape(-1)

print(b)
print(A[:3,:3])
print("gtsam sol", J[:,:].T@b)
print("gbp sol", gbp_graph.factors[i].factor.eta[:])

[0. 0. 0.]
[[1.e+00 0.e+00 0.e+00]
 [0.e+00 1.e+00 0.e+00]
 [0.e+00 0.e+00 1.e+04]]
gtsam sol [0. 0. 0.]
gbp sol [ 376.64272581 -499.67023461 5773.59001389]


In [22]:
for i in range(eta.shape[0]//3):
    if np.linalg.norm(eta[i*3:(i+1)*3 ] - eta2[i*3:(i+1)*3 ]) > 1e-8:
        print("difference at node ", i, np.linalg.norm(eta[i*3:(i+1)*3 ] - eta2[i*3:(i+1)*3 ]))

difference at node  117 5807.398017594845
difference at node  188 13393.68196975717
difference at node  261 13348.715205139699
difference at node  277 3684.3211862707244
difference at node  356 17134.671212332145
difference at node  372 25945.88051711741
difference at node  389 7934.229180012842
difference at node  419 5454.294341978028
difference at node  457 13773.926939454572
difference at node  502 1581.9189801718107


In [12]:
for i in range(eta2.shape[0]//3):
    print(eta2[i*3:(i+1)*3 ])


[0. 0. 0.]
[ -0.02689411  -0.02959865 -11.52967471]
[-3.71828932e-15 -9.83547241e-16  2.59104808e-12]
[ 1.34659325e-14 -2.06683925e-15 -1.09102707e-12]
[ 4.22621001e-15 -1.66371955e-14 -1.86512108e-12]
[-7.96493348e-15  1.01748642e-14  1.02160038e-12]
[-4.86218485e-15  1.79099058e-14 -1.07588589e-12]
[6.65754799e-15 1.63862505e-14 3.13412166e-12]
[ 1.74180877e-14 -5.42102003e-15 -4.93104231e-12]
[-6.70180732e-15  1.06047821e-14  1.86540258e-12]
[ 6.40912225e-15 -2.63059831e-15 -1.66066685e-12]
[-8.71024894e-15  1.73025668e-15  2.28078456e-12]
[-3.99159845e-02 -2.46728865e-03  1.09282415e+01]
[ 1.44388660e-14 -1.00542680e-15 -2.61185946e-13]
[ 5.73876213e-15 -9.91632874e-16 -1.04523825e-12]
[-1.31527867e-14  2.35958852e-15  1.11398586e-12]
[ 1.00326981e-14 -1.53791430e-14 -7.82458555e-13]
[-5.41966249e-14  7.05358206e-15 -1.06263325e-12]
[9.34065062e-16 5.26119721e-14 1.85881993e-13]
[2.22768450e-14 1.17621402e-15 7.66809826e-13]
[-1.18037217e-14 -1.21318634e-14  9.82057475e-13]
[-7.092

In [22]:
import numpy as np
import gtsam

def extract_between_A_b(graph, values, factor_index: int):
    """
    Linearize graph[factor_index] at `values`.
    Return (key_i, key_j, Ai, Aj, b) for a binary Pose2 factor.

    Works with python gtsam bindings: JacobianFactor.getA() has NO args.
    """
    f = graph.at(factor_index)
    print(1111, f"{f.measured()}")
    gf = f.linearize(values)              # returns a GaussianFactor (usually JacobianFactor)

    # keys in this factor
    keys = list(gf.keys())
    assert len(keys) == 2, f"not a binary factor? len(keys)={len(keys)}"

    # In python binding, gf.getA() returns the full A matrix (m x n)
    A = np.array(gf.getA(), dtype=float)  # for Pose2 between: (3,6)
    b = np.array(gf.getb(), dtype=float).reshape(-1)  # (3,)

    print(b)
    # For Pose2, each variable dim is 3
    di = 3
    dj = 3
    assert A.shape[1] == di + dj, f"unexpected A shape {A.shape}, expected 6 cols"

    Ai = A[:, :di]
    Aj = A[:, di:di+dj]

    lam = A.T@A
    print(np.array2string(lam, suppress_small=True, precision=6))

    def decode_key(k):
        s = gtsam.Symbol(k)
        return f"{chr(s.chr())}{s.index()}"
    print(f"Factor between keys: {decode_key(keys[0])} and {decode_key(keys[1])}")
    print(A.T@b)


extract_between_A_b(stats["graph"], stats["initial"], 100)

1111 (24.5145, 5.66018, 0.191303)

[ 1.42108547e-14 -0.00000000e+00 -5.55111512e-15]
[[     1.           -0.           -5.660181     -0.981757      0.190139
       0.      ]
 [    -0.            1.           24.514491     -0.190139     -0.981757
       0.      ]
 [    -5.660181     24.514491  10632.997924      0.895774    -25.143498
  -10000.      ]
 [    -0.981757     -0.190139      0.895774      1.            0.
       0.      ]
 [     0.190139     -0.981757    -25.143498      0.            1.
       0.      ]
 [     0.            0.       -10000.            0.            0.
   10000.      ]]
Factor between keys: x99 and x100
[-1.39516098e-14 -2.70203158e-15  5.67841224e-13  1.42108547e-14
  0.00000000e+00 -5.55111512e-13]


In [444]:
gfg = stats["graph"].linearize(stats["initial"])
A = np.array(gfg.hessian()[0])
b = np.array(gfg.hessian()[1])

eta, lam = basegraph.joint_distribution_inf()
u = np.concatenate([v.mu for v in basegraph.var_nodes])


total4 = []
for i in range(len(basegraph.var_nodes)):
    gt = np.asarray(basegraph.var_nodes[i].GT[0:3], dtype=float)
    mu = u[basegraph.var_nodes[i].dofs*i : basegraph.var_nodes[i].dofs*(i+1)]
    r = mu[0:2] - gt[0:2]
    total4.append(0.5 * (float(r.T @ r) ))#+  10000*(gt[2]- mu[2])**2)
print(np.sum(total4))

delta = np.linalg.solve(A,b)
def compose_SE2(pose_i, z_ij):
    """Given local measurement z_ij, propagate from pose_i to pose_j."""
    xi, yi, thi = pose_i
    dx, dy, dth = z_ij
    c, s = np.cos(thi), np.sin(thi)
    t_global = np.array([c*dx - s*dy, s*dx + c*dy])
    xj = xi + t_global[0]
    yj = yi + t_global[1]
    thj = wrap_angle(thi + dth)
    return np.array([xj, yj, thj], dtype=float)

for i in range(len(basegraph.var_nodes)):
    delta_i = delta[basegraph.var_nodes[i].dofs*i : basegraph.var_nodes[i].dofs*(i+1)]
    u_i = u[basegraph.var_nodes[i].dofs*i : basegraph.var_nodes[i].dofs*(i+1)]
    u[basegraph.var_nodes[i].dofs*i : basegraph.var_nodes[i].dofs*(i+1)] = compose_SE2(u_i, delta_i)

total4 = []
for i in range(len(basegraph.var_nodes)):
    gt = np.asarray(basegraph.var_nodes[i].GT[0:3], dtype=float)
    mu = u[basegraph.var_nodes[i].dofs*i : basegraph.var_nodes[i].dofs*(i+1)]
    r = mu[0:2] - gt[0:2]
    total4.append(0.5 * (float(r.T @ r) ))#+  10000*(gt[2]- mu[2])**2)
print(np.sum(total4))

41233.35115424025
4594.256256894979


In [397]:
key = gtsam.symbol('x', 1)
pose = stats["initial"].atPose2(key)

# 拆值
print(pose, stats["GT"][1])
print(basegraph.var_nodes[1].mu)

(24.6761, 3.89657, 0.223854)
 [24.46976093  5.1216013   0.20632481]
[24.67608574  3.89657445  0.22385359]


In [306]:
def values_to_numpy_pose2(values: gtsam.Values, N: int):
    """
    Convert GTSAM Values (Pose2 with keys x0..x{N-1})
    to numpy array of shape (N,3): [x, y, theta].
    """
    out = np.zeros((N, 3), dtype=float)
    for i in range(N):
        pose: gtsam.Pose2 = values.atPose2(symbol("x", i))
        out[i, 0] = pose.x()
        out[i, 1] = pose.y()
        out[i, 2] = pose.theta()
    return out

mu_opt = values_to_numpy_pose2(stats["result"], 512)


In [396]:
i= 228
print(basegraph.var_nodes[i].GT)
print(stats["GT"][i])
print(mu_opt[i])
print(np.linalg.solve(basegraph.var_nodes[i].belief.lam,basegraph.var_nodes[i].belief.eta))

[621.00845746 340.93067243   0.97091896]
[621.00845746 340.93067243   0.97091896]
[623.79235744 341.97663069   0.95902559]


LinAlgError: Singular matrix

In [211]:
(169.16995085-167.74333255)**2+(239.66912546-233.31367701)**2

42.426964774502444

In [135]:
(173.57635562-167.74333255)**2+(237.80885336-233.31367701)**2

54.23076855275157

In [395]:
total = []
for i in range(len(basegraph.var_nodes)):
    gt = np.asarray(basegraph.var_nodes[i].GT[0:3], dtype=float)

    key = gtsam.symbol('x', 1)

    pose = stats["initial"].atPose2(gtsam.symbol('x', i))
    arr = np.array([pose.x(), pose.y(), pose.theta()], dtype=float)
    r = arr[:2] - gt[0:2]
    total.append(0.5 *(float(r.T @ r) ))# +  10000*(gt[2]- mu_opt[i][2])**2  )

print(np.sum(total))

4449637.942617746


In [309]:
total2 = []
for i in range(len(basegraph.var_nodes)):
    gt = np.asarray(basegraph.var_nodes[i].GT[0:3], dtype=float)
    mu = np.linalg.solve(basegraph.var_nodes[i].belief.lam,basegraph.var_nodes[i].belief.eta)
    r = mu[0:2] - gt[0:2]
    total2.append(0.5 * (float(r.T @ r) ))#+  10000*(gt[2]- mu[2])**2)
print(2*np.sum(total2))

153800.42411176022


In [ ]:
N = 512
step=25
prob=0.05
radius=50 
prior_prop=0.02
prior_sigma=1
odom_sigma=1
layers = []


layers = init_layers(N=N, step_size=step, loop_prob=prob, loop_radius=radius, prior_prop=prior_prop, seed=2001)
pair_idx = 0



# Create GBP graph
gbp_graph = build_noisy_pose_graph(layers[0]["nodes"], layers[0]["edges"],
                                    prior_sigma=prior_sigma,
                                    odom_sigma=odom_sigma,
                                    seed=2001)
layers[0]["graph"] = gbp_graph
gbp_graph.num_undamped_iters = 0
gbp_graph.min_linear_iters = 2000
opts=[{"label":"base","value":"base"}]


kk = 10
k_next = 1
super_layer_idx = k_next*2 - 1
last = layers[-1]
super_nodes, super_edges, node_map = fuse_to_super_order(last["nodes"], last["edges"], int(kk or 8), super_layer_idx, tail_heavy=True)
# Ensure base graph has run at least once
layers[-1]["graph"].synchronous_iteration() 
layers.append({"name":f"super{k_next}", "nodes":super_nodes, "edges":super_edges, "node_map":node_map})
if super_layer_idx > 1:
    layers[super_layer_idx]["graph"] = build_super_graph(layers)
else:
    layers[super_layer_idx]["graph"] = build_super_graph(layers)



abs_layer_idx = 2
k = 1
last = layers[-1]
abs_nodes, abs_edges = copy_to_abs(last["nodes"], last["edges"], abs_layer_idx)
# Ensure super graph has run at least once
layers[-1]["graph"].synchronous_iteration() 
layers.append({"name":f"abs{k}", "nodes":abs_nodes, "edges":abs_edges})
layers[abs_layer_idx]["graph"], layers[abs_layer_idx]["Bs"], layers[abs_layer_idx]["ks"], layers[abs_layer_idx]["k2s"] = build_abs_graph(
    layers, r_reduced=2)



k_next = 2
super_layer_idx = k_next*2 - 1
last = layers[-1]
super_nodes, super_edges, node_map = fuse_to_super_order(last["nodes"], last["edges"], int(kk or 8), super_layer_idx, tail_heavy=True)
# Ensure super graph has run at least once
layers[-1]["graph"].synchronous_iteration() 
layers.append({"name":f"super{k_next}", "nodes":super_nodes, "edges":super_edges, "node_map":node_map})
if super_layer_idx > 1:
    layers[super_layer_idx]["graph"] = build_super_graph(layers)
else:
    layers[super_layer_idx]["graph"] = build_super_graph(layers)



abs_layer_idx = 4
k = 2
last = layers[-1]
abs_nodes, abs_edges = copy_to_abs(last["nodes"], last["edges"], abs_layer_idx)
# Ensure super graph has run at least once
layers[-1]["graph"].synchronous_iteration() 
layers.append({"name":f"abs{k}", "nodes":abs_nodes, "edges":abs_edges})
layers[abs_layer_idx]["graph"], layers[abs_layer_idx]["Bs"], layers[abs_layer_idx]["ks"], layers[abs_layer_idx]["k2s"] = build_abs_graph(
    layers, r_reduced=2)



"""
k_next = 3
super_layer_idx = k_next*2 - 1
last = layers[-1]
super_nodes, super_edges, node_map = fuse_to_super_order(last["nodes"], last["edges"], int(kk or 8), super_layer_idx, tail_heavy=True)
# Ensure super graph has run at least once
layers[-1]["graph"].synchronous_iteration() 
layers.append({"name":f"super{k_next}", "nodes":super_nodes, "edges":super_edges, "node_map":node_map})
if super_layer_idx > 1:
    layers[super_layer_idx]["graph"] = build_super_graph(layers)
else:
    layers[super_layer_idx]["graph"] = build_super_graph(layers)


abs_layer_idx = 6
k = 3
last = layers[-1]
abs_nodes, abs_edges = copy_to_abs(last["nodes"], last["edges"], abs_layer_idx)
# Ensure super graph has run at least once
layers[-1]["graph"].synchronous_iteration() 
layers.append({"name":f"abs{k}", "nodes":abs_nodes, "edges":abs_edges})
layers[abs_layer_idx]["graph"], layers[abs_layer_idx]["Bs"], layers[abs_layer_idx]["ks"], layers[abs_layer_idx]["k2s"] = build_abs_graph(layers)
"""

"""
k_next = 4
super_layer_idx = k_next*2 - 1
last = layers[-1]
super_nodes, super_edges, node_map = fuse_to_super_order(last["nodes"], last["edges"], int(kk or 8), super_layer_idx, tail_heavy=True)
# Ensure super graph has run at least once
layers[-1]["graph"].synchronous_iteration() 
layers.append({"name":f"super{k_next}", "nodes":super_nodes, "edges":super_edges, "node_map":node_map})
if super_layer_idx > 1:
    layers[super_layer_idx]["graph"] = build_super_graph(layers)
else:
    layers[super_layer_idx]["graph"] = build_super_graph(layers)


abs_layer_idx = 8
k = 4
last = layers[-1]
abs_nodes, abs_edges = copy_to_abs(last["nodes"], last["edges"], abs_layer_idx)
# Ensure super graph has run at least once
layers[-1]["graph"].synchronous_iteration() 
layers.append({"name":f"abs{k}", "nodes":abs_nodes, "edges":abs_edges})
layers[abs_layer_idx]["graph"], layers[abs_layer_idx]["Bs"], layers[abs_layer_idx]["ks"], layers[abs_layer_idx]["k2s"] = build_abs_graph(layers)
"""


"""
for i in range(2000):
    layers[1]["graph"].synchronous_iteration()
    layers[0]["graph"].synchronous_iteration()
    layers[2]["graph"].synchronous_iteration()
    #top_down_modify_super_graph(layers[:])
    #top_down_modify_base_and_abs_graph(layers[0:2])
    energy = energy_map(layers[0]["graph"], include_priors=True, include_factors=True)
    print(f"Iter {i+1:03d} | Energy = {energy:.6f}")
"""

vg = VGraph(layers)
energy_prev = 0
counter = 0
for _ in range(200):
    vg.layers = layers
    vg.r_reduced=3
    vg.eta_damping = 0
    vg.layers = vg.vloop()
    energy = energy_map(layers[0]["graph"], include_priors=True, include_factors=True)
    if np.abs(energy_prev-energy) < 1e-5:
        counter += 1
        if counter >= 2:
            break
    print(f"Iter {_+1:03d} | Energy = {energy:.6f}")

    #energy_prev = energy
refresh_gbp_results(layers)
